In [1]:
# 强制升级 vertexai 及其依赖
!pip install --upgrade google-cloud-aiplatform --quiet

In [2]:
import os
import zipfile
import time  # <--- 新增
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import storage
from google.api_core.exceptions import ResourceExhausted, ServiceUnavailable, InternalServerError # <--- 新增：用于精确捕获错误

# 1. 初始化
PROJECT_ID = "gen-lang-client-0371685655"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
model = GenerativeModel("gemini-2.5-pro")

# 2. 配置路径
BUCKET_NAME = 'xhs-humor-data'
SEARCH_KEYWORD = "妈的欧洲账本"
LOCAL_BASE_DIR = './raw_data'
EXTRACT_BASE_DIR = './extracted_results'

os.makedirs(LOCAL_BASE_DIR, exist_ok=True)
os.makedirs(EXTRACT_BASE_DIR, exist_ok=True)

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
all_blobs = storage_client.list_blobs(BUCKET_NAME)
target_zips = [b for b in all_blobs if SEARCH_KEYWORD in b.name and b.name.endswith('.zip')]

# --- 核心逻辑 A: 预扫描 ---
print("正在预扫描文件总量...")
total_media_files = 0
task_list = []

for zip_blob in target_zips:
    local_temp_zip = os.path.join(LOCAL_BASE_DIR, "scan_temp.zip")
    zip_blob.download_to_filename(local_temp_zip)
    with zipfile.ZipFile(local_temp_zip, 'r') as z:
        media_in_zip = [f for f in z.namelist() if f.lower().endswith(('.png', '.jpg', '.jpeg', '.mp4')) and not f.startswith('__MACOSX')]
        total_media_files += len(media_in_zip)
        task_list.append((zip_blob, media_in_zip))
    os.remove(local_temp_zip)

print(f"找到 {len(target_zips)} 个压缩包，共计 {total_media_files} 个媒体文件。\n")

# --- 核心逻辑 B: 增加断点续传 + 自动重试机制 ---
processed_count = 0

for zip_blob, media_files in task_list:
    subject_name = zip_blob.name.split('/')[-1].replace('.zip', '')
    subject_dir = os.path.join(EXTRACT_BASE_DIR, subject_name)
    os.makedirs(subject_dir, exist_ok=True)

    local_zip = os.path.join(LOCAL_BASE_DIR, "current_process.zip")
    zip_blob.download_to_filename(local_zip)
    
    with zipfile.ZipFile(local_zip, 'r') as z:
        z.extractall(subject_dir)
    
    for root, _, files in os.walk(subject_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.mp4')) and not file.startswith('._'):
                processed_count += 1
                
                # --- 检查云端是否存在结果 ---
                txt_file_name = f"{file}.txt"
                blob_path = f"extracted_data/{subject_name}/{txt_file_name}"
                check_blob = bucket.blob(blob_path)
                
                if check_blob.exists():
                    print(f"进度: [{(processed_count / total_media_files) * 100:.1f}%] | 跳过已处理: {file}")
                    continue
                
                # --- 准备数据 ---
                file_path = os.path.join(root, file)
                global_progress = (processed_count / total_media_files) * 100
                print(f"进度: [{global_progress:.1f}%] | 正在处理: {file}")
                
                # --- 核心修改区域：重试逻辑 ---
                max_retries = 5  # 最大重试次数
                retry_delay = 5  # 初始等待时间（秒）
                
                for attempt in range(max_retries + 1):
                    try:
                        with open(file_path, "rb") as f:
                            data = f.read()
                        mime_type = "video/mp4" if file.lower().endswith('.mp4') else "image/jpeg"
                        media_part = Part.from_data(data=data, mime_type=mime_type)
                        
                        prompt = "请提取并列出该媒体文件中的所有文字。如果是账本或清单，请保持其条目格式。直接输出内容。"
                        response = model.generate_content([prompt, media_part])
                        text_result = response.text.strip()
                        
                        # 保存本地
                        local_txt_path = os.path.join(subject_dir, txt_file_name)
                        with open(local_txt_path, "w", encoding="utf-8") as f_txt:
                            f_txt.write(text_result)
                        
                        # 上传云端
                        check_blob.upload_from_filename(local_txt_path)
                        
                        # 如果成功，跳出重试循环
                        break 

                    except (ResourceExhausted, ServiceUnavailable, InternalServerError) as e:
                        # 专门捕获 429 (配额耗尽) 和 503 (服务不可用)
                        if attempt < max_retries:
                            sleep_time = retry_delay * (2 ** attempt) # 指数退避: 5s, 10s, 20s...
                            print(f"  [Warning] 触发限流/服务繁忙 (429/5xx)。等待 {sleep_time} 秒后重试... (尝试 {attempt+1}/{max_retries})")
                            time.sleep(sleep_time)
                        else:
                            print(f"  [Error] {file} 重试多次失败，跳过。错误信息: {e}")
                    
                    except Exception as e:
                        # 捕获其他不可重试的错误（如文件损坏、格式不支持等）
                        print(f"  [Error] 处理 {file} 出现未知错误: {e}")
                        break # 不重试，直接处理下一个文件

    os.remove(local_zip)

print(f"\n任务处理完成！")

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

正在预扫描文件总量...
找到 11 个压缩包，共计 2506 个媒体文件。

进度: [0.0%] | 跳过已处理: 为了去欧洲决定结个婚_3.jpg
进度: [0.1%] | 跳过已处理: 为了去欧洲决定结个婚_10.jpg
进度: [0.1%] | 跳过已处理: 为了去欧洲决定结个婚_9.jpg
进度: [0.2%] | 跳过已处理: 为了去欧洲决定结个婚_8.jpg
进度: [0.2%] | 跳过已处理: 为了去欧洲决定结个婚_1.jpg
进度: [0.2%] | 跳过已处理: 为了去欧洲决定结个婚_14.jpg
进度: [0.3%] | 跳过已处理: 为了去欧洲决定结个婚_11.jpg
进度: [0.3%] | 跳过已处理: 为了去欧洲决定结个婚_12.jpg
进度: [0.4%] | 跳过已处理: 为了去欧洲决定结个婚_5.jpg
进度: [0.4%] | 跳过已处理: 为了去欧洲决定结个婚_2.jpg
进度: [0.4%] | 跳过已处理: 为了去欧洲决定结个婚_13.jpg
进度: [0.5%] | 跳过已处理: 为了去欧洲决定结个婚_7.jpg
进度: [0.5%] | 跳过已处理: 为了去欧洲决定结个婚_4.jpg
进度: [0.6%] | 跳过已处理: 为了去欧洲决定结个婚_6.jpg
进度: [0.6%] | 跳过已处理: 北欧旅行十大至暗时刻_1.jpg
进度: [0.6%] | 跳过已处理: 北欧旅行十大至暗时刻_10.jpg
进度: [0.7%] | 跳过已处理: 北欧旅行十大至暗时刻_4.jpg
进度: [0.7%] | 跳过已处理: 北欧旅行十大至暗时刻_12.jpg
进度: [0.8%] | 跳过已处理: 北欧旅行十大至暗时刻_8.jpg
进度: [0.8%] | 跳过已处理: 北欧旅行十大至暗时刻_9.jpg
进度: [0.8%] | 跳过已处理: 北欧旅行十大至暗时刻_14.jpg
进度: [0.9%] | 跳过已处理: 北欧旅行十大至暗时刻_5.jpg
进度: [0.9%] | 跳过已处理: 北欧旅行十大至暗时刻_6.jpg
进度: [1.0%] | 跳过已处理: 北欧旅行十大至暗时刻_13.jpg
进度: [1.0%] | 跳过已处理: 北欧旅行十大至暗时刻_15.jpg
进度: [1.0%] | 跳过已处理: 北欧旅行十